# Imports, config, session and utilities

In [75]:
from pathlib import Path
import sys

from snowflake.core import Root
from snowflake.snowpark.session import Session

In [76]:
current_file_path = Path().resolve()
sys.path.insert(0, str(current_file_path.parent))

In [77]:
# Custom python file import
# Import statement after adding parent file path to recognize src python files
from src.support_agent.config import get_settings
from src.support_agent.snowflake_client import create_snowpark_session

In [78]:
# Getting .env config and init snowflake session
settings = get_settings()
session = create_snowpark_session(settings)

# Build Search Service

Create and test Cortex Search service.

## Setup search service


- To setup curated tickets_cleaned, run scripts/process_data (can take one hour)
- Create search service pointing on curated table

In [ ]:
query_res = session.sql("SELECT * FROM PROJECT_DB.CURATED.TICKETS_CLEANED;").to_pandas()

In [67]:
query_res.head()

,SUBJECT,BODY,ANSWER,TYPE,QUEUE,PRIORITY,LANGUAGE,TAG_1,TAG_2,REWRITTEN_BODY,CLEANED_ANSWER
0,None,"Customer Support, we have received your inquir...","Greetings, we are grateful for reaching out to...",Request,IT Support,high,en,Feedback,Sales,Please provide information on techniques and t...,Our team employs a combination of social media...
1,None,"Customer Support, <br><br>I am writing to brin...",We will investigate the data analytics tool ma...,Problem,Customer Service,medium,en,Bug,Performance,Critical Issue: Data Analytics Tool Producing ...,Investigating data analytics tool malfunction....
2,Verbesserung von Digitalen Werkzeugen,Bitte kontaktieren Sie uns für die Optimierung...,Vielen Dank für Ihr Interesse an der Verbesser...,Change,IT Support,medium,de,Feedback,Feature,Anfrage zur Optimierung digitaler Tools für ve...,"Rufen Sie uns bitte unter <tel_num> an, um wei..."
3,Incorrect Fee Payment,"Incorrect subscription payment received, possi...","Apologies for the incorrect fee payment, pleas...",Problem,Billing and Payments,medium,de,Billing,Payment,Could you investigate an incorrect subscriptio...,Check account statement <acc_num> to resolve t...
4,Request for Details on Project Management Feat...,Is it possible to get more details on the cust...,We appreciate your interest in the customizati...,Request,Customer Service,low,en,Feature,Feedback,What customization options are available for p...,The platform offers customizable workflows and...


In [87]:
query_res["LANGUAGE"].value_counts().keys()

Index(['en', 'de'], dtype='object', name='LANGUAGE')

In [71]:
create_search_service_query = """
CREATE OR REPLACE CORTEX SEARCH SERVICE PROJECT_DB.SERVICES.support_tickets_search_service
  VECTOR INDEXES REWRITTEN_BODY (model='voyage-multilingual-2')
  ATTRIBUTES PRIORITY, TYPE, LANGUAGE
  WAREHOUSE = compute_wh
  TARGET_LAG = '1 day'
  AS (
    SELECT 
      SUBJECT,
      REWRITTEN_BODY,
      CLEANED_ANSWER,
      TYPE,
      QUEUE,
      PRIORITY,
      LANGUAGE,
      TAG_1,
      TAG_2
    FROM PROJECT_DB.CURATED.TICKETS_CLEANED
);
"""

query_res = session.sql(create_search_service_query).collect()
query_res

[Row(status='Cortex search service SUPPORT_TICKETS_SEARCH_SERVICE successfully created.')]

## Test search service with custom queries

In [72]:
root = Root(session)

# fetch service
my_service = (root
    .databases["PROJECT_DB"]
    .schemas["SERVICES"]
    .cortex_search_services["support_tickets_search_service"]
)

# query service
resp = my_service.search(
    query="Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement.",
    columns=["REWRITTEN_BODY", "CLEANED_ANSWER", "TYPE"],
    limit=5
)
print(resp.to_json())

{"results": [{"CLEANED_ANSWER": "Our team employs a combination of social media marketing, content creation, and email campaigns to enhance brand awareness and engagement. We utilize analytics tools to track performance and make data-driven decisions.", "@scores": {"cosine_similarity": 0.77519596}, "REWRITTEN_BODY": "Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement.", "TYPE": "Request"}, {"CLEANED_ANSWER": "Our team employs a combination of social media marketing, content creation, and email campaigns to increase brand awareness and engagement. We leverage analytics tools to track performance and make data-driven decisions.", "@scores": {"cosine_similarity": 0.75207335}, "REWRITTEN_BODY": "Please provide information about techniques and tools used for increasing brand awareness and digital engagement. Seeking recommendations for digital strategy implementation to support brand growt

In [73]:
import json
json_resp = json.loads(resp.to_json())

In [74]:
json_resp["results"]

[{'CLEANED_ANSWER': 'Our team employs a combination of social media marketing, content creation, and email campaigns to enhance brand awareness and engagement. We utilize analytics tools to track performance and make data-driven decisions.',
  '@scores': {'cosine_similarity': 0.77519596},
  'REWRITTEN_BODY': 'Please provide information on techniques and tools for digital brand growth strategies, specifically focused on increasing brand awareness and engagement.',
  'TYPE': 'Request'},
 {'CLEANED_ANSWER': 'Our team employs a combination of social media marketing, content creation, and email campaigns to increase brand awareness and engagement. We leverage analytics tools to track performance and make data-driven decisions.',
  '@scores': {'cosine_similarity': 0.75207335},
  'REWRITTEN_BODY': 'Please provide information about techniques and tools used for increasing brand awareness and digital engagement. Seeking recommendations for digital strategy implementation to support brand growth

Same body but cosine_similarity =0.77, ...

# Close session 

In [6]:
session.close()